# Critical Heat Flux Prediction (Optimized & Fast Pipeline)
Predicting Critical Heat Flux (CHF) using physical feature engineering and ExtraTrees regression.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.ensemble import ExtraTreesRegressor

import warnings
warnings.filterwarnings("ignore")

In [ ]:
# Download and load dataset
!pip install --upgrade --quiet gdown
!gdown 1K_guA0w-8Z9EC2F-A-pBXOVVCRxkNuQm -O dataset.zip
!unzip -o dataset.zip -d dataset

df = pd.read_csv("dataset/dataset/Data_CHF_Zhao_2020_ATE.csv")
if "id" in df.columns:
    df = df.drop(["id"], axis=1)

df.columns = ["author", "geometry", "pressure", "mass_flux", "exit_concentration", 
              "equivalent_diameter", "hydraulic_diameter", "channel_length", "exp_critical_heat_flux"]

In [ ]:
# Physical feature engineering
df["aspect_ratio"] = df["channel_length"] / (df["equivalent_diameter"] + 1e-5)
df["diameter_ratio"] = df["equivalent_diameter"] / (df["hydraulic_diameter"] + 1e-5)
df["flux_quality_prod"] = df["mass_flux"] * df["exit_concentration"]
df["press_flux_prod"] = df["pressure"] * df["mass_flux"]

# One-Hot Encoding for categorical features
df_encoded = pd.get_dummies(df, columns=["author", "geometry"], drop_first=True)
X = df_encoded.drop("exp_critical_heat_flux", axis=1)
y = df_encoded["exp_critical_heat_flux"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Fast model training (ExtraTrees Ensemble)
model = ExtraTreesRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
r2 = r2_score(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)

print("="*40)
print(f"R2 Score:                {r2:.5f}")
print(f"Mean Square Error (MSE): {mse:.5f}")
print(f"Root Mean Square Error:  {rmse:.5f}")
print("="*40)

In [ ]:
# Scatter plot: Actual vs Predicted
plt.figure(figsize=(6, 5))
plt.scatter(y_test, y_pred, alpha=0.5, color="crimson")
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], "k--", lw=2)
plt.xlabel("Actual Values (MW/m²)")
plt.ylabel("Predicted Values (MW/m²)")
plt.title(f"Model Performance (R² = {r2:.3f})")
plt.grid(True)
plt.show()